# Data overview

A first look at the data in `data/csv/`: sizes, rows and columns, a peek at the main tables, missing values and time coverage.

See `docs/DATA_GUIDE.md` for what each file contains. Section 2 reads every file in chunks (the play-by-play files are several GB), so it takes a few minutes; the rest is fast.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "csv"


def read(rel: str, **kwargs) -> pd.DataFrame:
    return pd.read_csv(DATA / rel, low_memory=False, **kwargs)


print("Data folder:", DATA)

Data folder: D:\challenge-two\data\csv


## 1. Size on disk

In [2]:
files = pd.DataFrame(
    [{"source": p.relative_to(DATA).parts[0],
      "file": p.relative_to(DATA).as_posix(),
      "size_mb": p.stat().st_size / 1e6}
     for p in sorted(DATA.rglob("*.csv"))]
)

by_source = (files.groupby("source")
             .agg(files=("file", "count"), size_mb=("size_mb", "sum"))
             .sort_values("size_mb", ascending=False))
by_source["size_gb"] = (by_source["size_mb"] / 1e3).round(2)

print(f"Total: {len(files)} files, {files.size_mb.sum() / 1e3:.2f} GB")
by_source[["files", "size_gb"]]

Total: 56 files, 9.27 GB


,files,size_gb
source,,
kaggle_eoinamoore_historical_nba,11,6.73
kaggle_wyattowalsh_basketball,16,2.19
sportsdataverse_injuries,9,0.24
kaggle_nathanlauga_nba_games,5,0.11
boxscorelab,15,0.00


In [3]:
print("10 largest files")
files.sort_values("size_mb", ascending=False).head(10).round(1)

10 largest files


,source,file,size_mb
19,kaggle_eoinamoore_historical_nba,kaggle_eoinamoore_historical_nba/PlayByPlay.csv,5790.6
41,kaggle_wyattowalsh_basketball,kaggle_wyattowalsh_basketball/play_by_play.csv,2140.4
22,kaggle_eoinamoore_historical_nba,kaggle_eoinamoore_historical_nba/PlayerStatist...,456.3
21,kaggle_eoinamoore_historical_nba,kaggle_eoinamoore_historical_nba/PlayerStatist...,391.3
27,kaggle_nathanlauga_nba_games,kaggle_nathanlauga_nba_games/games_details.csv,93.1
54,sportsdataverse_injuries,sportsdataverse_injuries/injury_report_2026.csv,66.8
25,kaggle_eoinamoore_historical_nba,kaggle_eoinamoore_historical_nba/TeamStatistic...,38.4
53,sportsdataverse_injuries,sportsdataverse_injuries/injury_report_2025.csv,37.9
50,sportsdataverse_injuries,sportsdataverse_injuries/injury_report_2022.csv,36.4
24,kaggle_eoinamoore_historical_nba,kaggle_eoinamoore_historical_nba/TeamStatistic...,36.1


## 2. Rows and columns

Every file is read in chunks of 500,000 rows, counting rows and missing values per column.

In [4]:
def profile(path: Path) -> pd.DataFrame:
    """One row per column: missing values over the whole file."""
    rows, nulls = 0, None
    for chunk in pd.read_csv(path, chunksize=500_000, dtype=str, keep_default_na=False, na_values=[""]):
        rows += len(chunk)
        n = chunk.isna().sum()
        nulls = n if nulls is None else nulls + n
    if nulls is None:  # header only
        nulls = pd.Series(0, index=pd.read_csv(path, nrows=0).columns)
    return pd.DataFrame({"column": nulls.index, "n_null": nulls.values, "rows": rows})


col_profile = pd.concat(
    [profile(DATA / f).assign(file=f) for f in files.file], ignore_index=True
)
col_profile["pct_null"] = (100 * col_profile.n_null / col_profile.rows.where(col_profile.rows > 0)).fillna(0).round(2)

per_file = (col_profile.groupby("file")
            .agg(rows=("rows", "first"), n_cols=("column", "count"), avg_pct_null=("pct_null", "mean"))
            .join(files.set_index("file")[["source", "size_mb"]]))

print(f"Total rows: {per_file.rows.sum():,}")
print(f"Total columns: {per_file.n_cols.sum():,}")
(per_file.groupby("source")
         .agg(files=("rows", "count"), rows=("rows", "sum"), max_cols=("n_cols", "max"))
         .sort_values("rows", ascending=False))

Total rows: 38,025,531
Total columns: 1,451


,files,rows,max_cols
source,,,
kaggle_eoinamoore_historical_nba,11,21697604,110
kaggle_wyattowalsh_basketball,16,14054248,55
sportsdataverse_injuries,9,1347766,17
kaggle_nathanlauga_nba_games,5,912879,29
boxscorelab,15,13034,32


In [5]:
print("Rows and columns per file")
per_file[["rows", "n_cols", "size_mb"]].sort_values("rows", ascending=False).round(1)

Rows and columns per file


,rows,n_cols,size_mb
file,,,
kaggle_eoinamoore_historical_nba/PlayByPlay.csv,18864106,89,5790.6
kaggle_wyattowalsh_basketball/play_by_play.csv,13585539,34,2140.4
kaggle_eoinamoore_historical_nba/PlayerStatistics.csv,1677321,40,391.3
kaggle_eoinamoore_historical_nba/PlayerStatisticsExtended.csv,844433,110,456.3
kaggle_nathanlauga_nba_games/games_details.csv,668628,29,93.1
sportsdataverse_injuries/injury_report_2026.csv,376069,17,66.8
sportsdataverse_injuries/injury_report_2025.csv,215095,17,37.9
kaggle_nathanlauga_nba_games/ranking.csv,210342,13,15.4
sportsdataverse_injuries/injury_report_2022.csv,206713,17,36.4


## 3. A peek at the main tables

In [6]:
games = read("kaggle_eoinamoore_historical_nba/Games.csv", parse_dates=["gameDateTimeEst"])
print(f"Games (eoinamoore): one row per game, {games.shape[0]:,} rows x {games.shape[1]} columns")
games.sort_values("gameDateTimeEst", ascending=False).head()

Games (eoinamoore): one row per game, 73,507 rows x 23 columns


,gameId,gameDateTimeEst,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,winner,gameType,gameSubtype,gameLabel,gameSubLabel,seriesGameNumber,attendance,arenaId,arenaName,arenaCity,arenaState,officials,gameDate
0,22600216,2026-11-10 23:00:00,Portland,Trail Blazers,1610612757,Oklahoma City,Thunder,1610612760,123,103,1610612757,Regular Season,NaN,NaN,NaN,NaN,19367.0,51,Moda Center,Portland,OR,"Curtis Blair, Eric Dalen, Brandon Schwab",2026-11-10 23:00:00
1,22600215,2026-11-10 22:00:00,Sacramento,Kings,1610612758,Orlando,Magic,1610612753,100,107,1610612753,Regular Season,NaN,NaN,NaN,NaN,13573.0,609,Golden 1 Center,Sacramento,CA,"Bill Kennedy, Kevin Cutler, Matt Myers",2026-11-10 22:00:00
2,22600213,2026-11-10 21:00:00,Phoenix,Suns,1610612756,LA,Clippers,1610612746,133,110,1610612756,Regular Season,NaN,NaN,NaN,NaN,17071.0,1000159,Mortgage Matchup Center,Phoenix,AZ,"Eric Lewis, Mitchell Ervin, Brent Haskill",2026-11-10 21:00:00
3,22600214,2026-11-10 21:00:00,Utah,Jazz,1610612762,Miami,Heat,1610612748,109,115,1610612748,Regular Season,NaN,NaN,NaN,NaN,17867.0,1000106,Delta Center,Salt Lake City,UT,"Sean Corbin, Scott Foster, JP Primm",2026-11-10 21:00:00
4,22600212,2026-11-10 20:00:00,Philadelphia,76ers,1610612755,Boston,Celtics,1610612738,101,123,1610612738,Regular Season,NaN,NaN,NaN,NaN,21497.0,1000152,Xfinity Mobile Arena,Philadelphia,PA,"James Capers, Ed Malloy, Intae Hwang",2026-11-10 20:00:00


In [7]:
team_stats = read("kaggle_eoinamoore_historical_nba/TeamStatistics.csv", parse_dates=["gameDateTimeEst"])
print(f"TeamStatistics (eoinamoore): one row per team per game, {team_stats.shape[0]:,} rows x {team_stats.shape[1]} columns")
team_stats.sort_values("gameDateTimeEst", ascending=False).head()

TeamStatistics (eoinamoore): one row per team per game, 147,016 rows x 59 columns


,gameId,gameDateTimeEst,teamCity,teamName,teamId,opponentTeamCity,opponentTeamName,opponentTeamId,home,win,teamScore,opponentScore,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,fieldGoalsPercentage,threePointersAttempted,threePointersMade,threePointersPercentage,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,...,q4Points,benchPoints,biggestLead,biggestScoringRun,leadChanges,pointsFastBreak,pointsFromTurnovers,pointsInThePaint,pointsSecondChance,timesTied,timeoutsRemaining,seasonWins,seasonLosses,coachId,gameType,gameLabel,gameSubLabel,seriesGameNumber,seed,reboundsTeam,turnoversTeam,ot1Points,ot2Points,otAllPoints,gameDate
0,22600216,2026-11-10 23:00:00,Portland,Trail Blazers,1610612757,Oklahoma City,Thunder,1610612760,1,1.0,123,103,26.0,4.0,4.0,91.0,40.0,0.440,50.0,19.0,0.380,26.0,24.0,0.923,33.0,...,26.0,38.0,22.0,9.0,14.0,13.0,11.0,34.0,15.0,14.0,1.0,7.0,3.0,NaN,Regular Season,NaN,NaN,NaN,NaN,9.0,1.0,NaN,NaN,NaN,2026-11-10 23:00:00
1,22600216,2026-11-10 23:00:00,Oklahoma City,Thunder,1610612760,Portland,Trail Blazers,1610612757,0,0.0,103,123,20.0,4.0,4.0,84.0,37.0,0.440,22.0,8.0,0.364,27.0,21.0,0.778,34.0,...,23.0,32.0,5.0,8.0,14.0,10.0,8.0,40.0,6.0,14.0,1.0,8.0,3.0,NaN,Regular Season,NaN,NaN,NaN,NaN,9.0,0.0,NaN,NaN,NaN,2026-11-10 23:00:00
2,22600215,2026-11-10 22:00:00,Sacramento,Kings,1610612758,Orlando,Magic,1610612753,1,0.0,100,107,19.0,7.0,8.0,75.0,33.0,0.440,34.0,16.0,0.471,22.0,18.0,0.818,30.0,...,24.0,15.0,5.0,18.0,5.0,11.0,18.0,28.0,8.0,4.0,0.0,1.0,10.0,NaN,Regular Season,NaN,NaN,NaN,NaN,9.0,1.0,NaN,NaN,NaN,2026-11-10 22:00:00
3,22600215,2026-11-10 22:00:00,Orlando,Magic,1610612753,Sacramento,Kings,1610612758,0,1.0,107,100,21.0,4.0,9.0,90.0,40.0,0.444,37.0,10.0,0.270,18.0,17.0,0.944,36.0,...,23.0,20.0,14.0,10.0,5.0,17.0,33.0,58.0,15.0,4.0,1.0,3.0,7.0,NaN,Regular Season,NaN,NaN,NaN,NaN,7.0,2.0,NaN,NaN,NaN,2026-11-10 22:00:00
4,22600213,2026-11-10 21:00:00,Phoenix,Suns,1610612756,LA,Clippers,1610612746,1,1.0,133,110,32.0,1.0,4.0,88.0,49.0,0.557,43.0,22.0,0.512,20.0,13.0,0.650,40.0,...,34.0,81.0,32.0,15.0,4.0,10.0,23.0,44.0,24.0,2.0,1.0,4.0,6.0,NaN,Regular Season,NaN,NaN,NaN,NaN,9.0,1.0,NaN,NaN,NaN,2026-11-10 21:00:00


In [8]:
print("PlayerStatistics (eoinamoore): one row per player per game (first 5 rows)")
read("kaggle_eoinamoore_historical_nba/PlayerStatistics.csv", nrows=5)

PlayerStatistics (eoinamoore): one row per player per game (first 5 rows)


,firstName,lastName,personId,gameId,gameDateTimeEst,playerteamCity,playerteamName,opponentteamCity,opponentteamName,gameType,gameLabel,gameSubLabel,seriesGameNumber,win,home,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,fieldGoalsPercentage,threePointersAttempted,threePointersMade,threePointersPercentage,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,playerteamId,opponentteamId,comment,startingPosition,gameDate
0,Chris,Youngblood,1642959,22600216,2026-11-10 23:00:00,Portland,Trail Blazers,Oklahoma City,Thunder,Regular Season,NaN,NaN,NaN,1,1,13.766667,8,0,0,1,5,2,0.400,4,2,0.500,2,2,1.000,1,0,1,2,0,0,1610612757,1610612760,NaN,NaN,2026-11-10 23:00:00
1,Robert,Williams III,1629057,22600216,2026-11-10 23:00:00,Portland,Trail Blazers,Oklahoma City,Thunder,Regular Season,NaN,NaN,NaN,1,1,14.550000,6,0,0,0,3,3,1.000,0,0,0.000,0,0,0.000,2,2,4,2,1,5,1610612757,1610612760,NaN,NaN,2026-11-10 23:00:00
2,Damian,Lillard,203081,22600216,2026-11-10 23:00:00,Portland,Trail Blazers,Oklahoma City,Thunder,Regular Season,NaN,NaN,NaN,1,1,23.500000,25,3,0,1,15,8,0.533,10,4,0.400,5,5,1.000,3,0,3,2,0,20,1610612757,1610612760,NaN,G,2026-11-10 23:00:00
3,Jared,McCain,1642272,22600216,2026-11-10 23:00:00,Oklahoma City,Thunder,Portland,Trail Blazers,Regular Season,NaN,NaN,NaN,0,0,31.366667,21,3,0,0,12,6,0.500,7,4,0.571,6,5,0.833,3,0,3,2,1,-14,1610612760,1610612757,NaN,G,2026-11-10 23:00:00
4,Micah,Peavy,1642877,22600216,2026-11-10 23:00:00,Oklahoma City,Thunder,Portland,Trail Blazers,Regular Season,NaN,NaN,NaN,0,0,20.966667,0,0,0,1,4,0,0.000,2,0,0.000,2,0,0.000,1,1,2,1,0,-11,1610612760,1610612757,NaN,NaN,2026-11-10 23:00:00


In [9]:
injuries = pd.concat(
    [pd.read_csv(p, low_memory=False) for p in sorted((DATA / "sportsdataverse_injuries").glob("*.csv"))],
    ignore_index=True,
)
print(f"Injury reports (sportsdataverse): one row per player per report, {injuries.shape[0]:,} rows x {injuries.shape[1]} columns")
injuries.dropna(subset=["report_player_name"]).tail()

Injury reports (sportsdataverse): one row per player per report, 1,347,766 rows x 17 columns


,season,report_datetime,game_date,game_time,report_matchup,away_team_abbr,home_team_abbr,team,report_player_name,current_status,reason,category,previous_status,game_id,team_id,person_id,match_method
1347753,2027,2026-11-11T00:00:00.000000,2026-11-11,07:00 (ET),WAS@IND,WAS,IND,Indiana Pacers,"Sheppard, Ben",Out,Injury/Illness - Left Meniscus Surgery,NaN,NaN,22600217.0,1.610613e+09,1641767.0,exact
1347754,2027,2026-11-11T00:00:00.000000,2026-11-11,07:00 (ET),WAS@IND,WAS,IND,Washington Wizards,"Beekman, Reece",Out,Injury/Illness - Right AC Joint Injury,NaN,NaN,22600217.0,1.610613e+09,1641736.0,exact
1347755,2027,2026-11-11T00:00:00.000000,2026-11-11,07:00 (ET),WAS@IND,WAS,IND,Washington Wizards,"Champagnie, Justin",Out,Injury/Illness - Right Foot Surgery,NaN,NaN,22600217.0,1.610613e+09,1630551.0,exact
1347756,2027,2026-11-11T00:00:00.000000,2026-11-11,07:00 (ET),WAS@IND,WAS,IND,Washington Wizards,"Vukcevic, Tristan",Out,Injury/Illness - Left Hip Impingement,NaN,NaN,22600217.0,1.610613e+09,1641774.0,exact
1347757,2027,2026-11-11T00:00:00.000000,2026-11-11,07:00 (ET),WAS@IND,WAS,IND,Washington Wizards,"Watkins, Jamir",Out,Injury/Illness - Left ACL Repair / Surgery,NaN,NaN,22600217.0,1.610613e+09,1642364.0,exact


In [10]:
print("Team standings 2025-26 (boxscorelab)")
read("boxscorelab/2025-26-team-standings.csv").sort_values("win_pct", ascending=False)

Team standings 2025-26 (boxscorelab)


,season,team,team_name,wins,losses,win_pct,games,points_pg,opponent_points_pg,off_rating_est,def_rating_est,net_rating,sos,adjusted_net_rating,pace
20,2025-26,OKC,Oklahoma City Thunder,64,18,0.7805,82,119.0244,107.8780,116.0221,105.2209,10.8012,-0.1241,10.6771,102.6178
26,2025-26,SAS,San Antonio Spurs,62,20,0.7561,82,119.8293,111.5244,116.6251,108.5502,8.0749,-0.0377,8.0372,102.7637
8,2025-26,DET,Detroit Pistons,60,22,0.7317,82,117.7683,109.6098,114.5281,106.5150,8.0130,-0.6391,7.3739,102.9310
2,2025-26,BOS,Boston Celtics,56,26,0.6829,82,114.8537,107.1585,117.2222,109.3338,7.8884,-0.3463,7.5421,97.9912
7,2025-26,DEN,Denver Nuggets,54,28,0.6585,82,122.0732,116.9268,119.5929,114.5420,5.0510,-0.1870,4.8640,102.0924
13,2025-26,LAL,Los Angeles Lakers,53,29,0.6463,82,116.3415,114.5854,115.3937,113.6945,1.6992,-0.1005,1.5987,100.8088
19,2025-26,NYK,New York Knicks,53,29,0.6463,82,116.4512,110.1220,116.6670,110.3521,6.3149,-0.2886,6.0263,99.7980
10,2025-26,HOU,Houston Rockets,52,30,0.6341,82,115.2317,110.0122,115.1855,109.9533,5.2322,-0.3824,4.8498,100.1185
5,2025-26,CLE,Cleveland Cavaliers,52,30,0.6341,82,119.5244,115.4146,116.1020,112.1569,3.9451,-0.3296,3.6155,102.9734
17,2025-26,MIN,Minnesota Timberwolves,49,33,0.5976,82,118.0000,114.6463,113.7421,110.7228,3.0193,-0.2917,2.7276,103.7324


## 4. Columns and types

Columns of `TeamStatistics.csv`, a likely base table for game prediction.

In [11]:
team_stats.dtypes.to_frame("dtype").assign(
    pct_null=(team_stats.isna().mean() * 100).round(2),
    example=team_stats.iloc[0],
)

,dtype,pct_null,example
gameId,int64,0.00,22600216
gameDateTimeEst,datetime64[us],0.00,2026-11-10 23:00:00
teamCity,str,0.01,Portland
teamName,str,0.00,Trail Blazers
teamId,int64,0.00,1610612757
opponentTeamCity,str,0.01,Oklahoma City
opponentTeamName,str,0.00,Thunder
opponentTeamId,int64,0.00,1610612760
home,int64,0.00,1
win,float64,0.00,1.0


In [12]:
team_stats.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
gameId,147016.0,25766341.86779,10300001.0,21300852.75,26100229.5,28700287.25,62500001.0,6422467.417475
gameDateTimeEst,147016,1997-03-29 17:49:12.943081,1946-11-26 19:00:00,1983-01-26 20:00:00,1999-11-10 19:45:00,2013-03-18 19:07:30,2026-11-10 23:00:00,NaN
teamId,147016.0,1610514154.716194,0.0,1610612744.0,1610612751.0,1610612758.0,1610616865.0,12601202.043771
opponentTeamId,147016.0,1610514154.716194,0.0,1610612744.0,1610612751.0,1610612758.0,1610616865.0,12601202.043771
home,147016.0,0.500007,0.0,0.0,1.0,1.0,1.0,0.500002
win,147014.0,0.5,0.0,0.0,0.5,1.0,1.0,0.500002
teamScore,147016.0,104.291254,0.0,95.0,104.0,114.0,186.0,14.266858
opponentScore,147016.0,104.291254,0.0,95.0,104.0,114.0,186.0,14.266858
assists,147012.0,17.690808,0.0,12.0,21.0,26.0,89.0,11.061157
blocks,147012.0,3.615419,0.0,0.0,3.0,6.0,23.0,3.175336


## 5. Missing values

In [13]:
tables = {
    "Games": games,
    "TeamStatistics": team_stats,
    "Players": read("kaggle_eoinamoore_historical_nba/Players.csv"),
    "Injuries": injuries,
}
for name, df in tables.items():
    pct = (df.isna().mean() * 100).round(2)
    missing = pct[pct > 0].sort_values(ascending=False)
    print(f"\n{name}: {len(missing)} of {df.shape[1]} columns have missing values")
    print(missing.head(10).to_string())


Games: 11 of 23 columns have missing values
gameSubtype         99.87
gameSubLabel        99.52
attendance          97.80
arenaState          97.80
officials           97.79
arenaCity           97.79
arenaName           97.79
gameLabel           94.38
seriesGameNumber    92.08
awayteamCity         0.01

TeamStatistics: 31 of 59 columns have missing values
coachId             100.00
ot2Points            99.25
ot1Points            94.67
otAllPoints          94.66
seed                 94.65
seriesGameNumber     93.96
gameSubLabel         93.69
gameLabel            93.55
q3Points              5.37
q4Points              5.37

Players: 19 of 20 columns have missing values
jersey           42.93
toYear           28.71
bodyWeightLbs    24.08
heightInches     23.97
school           21.00
country          20.89
fromYear         20.32
draftNumber      19.87
draftRound       19.02
dleagueFlag      18.04



Injuries: 9 of 17 columns have missing values
category              97.26
previous_status       96.44
person_id             10.55
match_method          10.55
report_player_name     9.35
current_status         9.35
game_id                0.63
reason                 0.01
team_id                0.01


In [14]:
print("Files with the most empty cells (average % null per column)")
per_file.avg_pct_null.sort_values(ascending=False).head(10).round(1)

Files with the most empty cells (average % null per column)


file
kaggle_eoinamoore_historical_nba/PlayByPlay.csv             66.8
kaggle_wyattowalsh_basketball/draft_combine_stats.csv       51.4
kaggle_eoinamoore_historical_nba/Games.csv                  38.0
kaggle_wyattowalsh_basketball/play_by_play.csv              36.2
kaggle_wyattowalsh_basketball/line_score.csv                31.9
kaggle_eoinamoore_historical_nba/LeagueSchedule24_25.csv    30.2
kaggle_eoinamoore_historical_nba/LeagueSchedule26_27.csv    22.4
kaggle_eoinamoore_historical_nba/LeagueSchedule25_26.csv    21.2
kaggle_nathanlauga_nba_games/games_details.csv              19.6
kaggle_wyattowalsh_basketball/game_summary.csv              19.6
Name: avg_pct_null, dtype: float64

In [15]:
print("Columns that are always empty")
col_profile[(col_profile.rows > 0) & (col_profile.n_null == col_profile.rows)][["file", "column"]].reset_index(drop=True)

Columns that are always empty


,file,column
0,boxscorelab/2022-23-player-season-totals.csv,position
1,boxscorelab/2023-24-player-season-totals.csv,position
2,kaggle_eoinamoore_historical_nba/LeagueSchedul...,seriesGameNumber
3,kaggle_eoinamoore_historical_nba/PlayByPlay.csv,gameType
4,kaggle_eoinamoore_historical_nba/TeamStatistic...,coachId
5,sportsdataverse_injuries/injury_report_2021.csv,category
6,sportsdataverse_injuries/injury_report_2021.csv,previous_status
7,sportsdataverse_injuries/injury_report_2022.csv,category
8,sportsdataverse_injuries/injury_report_2022.csv,previous_status
9,sportsdataverse_injuries/injury_report_2023.csv,category


## 6. Coverage over time

In [16]:
print("Games by type (eoinamoore)")
games.gameType.value_counts()

Games by type (eoinamoore)


gameType
Regular Season        66929
Playoffs               4526
Preseason              1940
NBA Emirates Cup         66
Play-in Tournament       37
All-Star Game             7
Emirates NBA Cup          1
NBA Cup                   1
Name: count, dtype: int64

In [17]:
print("Games per decade and type")
pd.crosstab(games.gameDateTimeEst.dt.year // 10 * 10, games.gameType).rename_axis("decade")

Games per decade and type


gameType,All-Star Game,Emirates NBA Cup,NBA Cup,NBA Emirates Cup,Play-in Tournament,Playoffs,Preseason,Regular Season
decade,,,,,,,,
1940,0,0,0,0,0,4,0,172
1950,0,0,0,0,0,205,0,2522
1960,0,0,0,0,0,298,0,4025
1970,0,0,0,0,0,453,0,7698
1980,0,0,0,0,0,619,0,9418
1990,0,0,0,0,0,716,0,10982
2000,0,0,0,0,0,810,597,12146
2010,0,0,0,0,0,831,859,12089
2020,7,1,1,66,37,590,484,7877


In [18]:
ext = read("kaggle_eoinamoore_historical_nba/TeamStatisticsExtended.csv",
           usecols=["gameDateTimeEst"], parse_dates=["gameDateTimeEst"])
coverage = pd.DataFrame({
    "team_games": team_stats.gameDateTimeEst.dt.year.value_counts(),
    "extended": ext.gameDateTimeEst.dt.year.value_counts(),
}).fillna(0).astype(int).sort_index()
print("Team-game rows per year: basic stats vs extended (advanced) stats")
coverage[(coverage.index % 5 == 0) | (coverage.index >= 2022)]

Team-game rows per year: basic stats vs extended (advanced) stats

,team_games,extended
gameDateTimeEst,,
1950,332,0
1955,612,0
1960,664,0
1965,768,0
1970,1362,0
1975,1524,0
1980,1948,0
1985,2030,0
1990,2370,0


## 7. Date range per source

In [19]:
def date_range(name, frames, col):
    dates = pd.to_datetime(pd.concat([f[col] for f in frames]), errors="coerce", format="mixed")
    return name, dates.min().date(), dates.max().date()


boxscore = [pd.read_csv(p, usecols=["game_date"]) for p in (DATA / "boxscorelab").glob("*-team-game-results.csv")]
pd.DataFrame(
    [
        date_range("eoinamoore Games", [games], "gameDateTimeEst"),
        date_range("wyattowalsh game", [read("kaggle_wyattowalsh_basketball/game.csv", usecols=["game_date"])], "game_date"),
        date_range("nathanlauga games", [read("kaggle_nathanlauga_nba_games/games.csv", usecols=["GAME_DATE_EST"])], "GAME_DATE_EST"),
        date_range("injury reports", [injuries], "game_date"),
        date_range("boxscorelab results", boxscore, "game_date"),
    ],
    columns=["source", "first", "last"],
)

,source,first,last
0,eoinamoore Games,1946-11-26,2026-11-10
1,wyattowalsh game,1946-11-01,2023-06-12
2,nathanlauga games,2003-10-05,2022-12-22
3,injury reports,2018-12-17,2026-11-12
4,boxscorelab results,2022-10-18,2026-11-10
